# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for exploring and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))
print("License:", getattr(metadata, 'license', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata model. All references use entity `@id`s.

We'll print the available `RecordSet` objects and their `@id`, then inspect their fields.

In [ ]:
# List all RecordSets and their metadata
for record_set in metadata.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', 'N/A')}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    Field @id: {field.id} -- Name: {getattr(field, 'name', '')}")
        if hasattr(field, 'columns') and field.columns:
            for column in field.columns:
                print(f"      Column @id: {column.id} -- Name: {getattr(column, 'name', '')}")
    print("\n")

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis. Use the RecordSet and Field `@id`s identified above.

Below, we extract all primary record sets (by `@id`) and convert their records to pandas DataFrames.

In [ ]:
# Extract data from each record set by @id

# Collect all RecordSet @id values
record_sets_ids = [record_set.id for record_set in metadata.record_sets]
dataframes = {}

# Preview record set IDs
print("Record sets in the dataset:")
for idx, rs_id in enumerate(record_sets_ids):
    print(f"  {idx}: {rs_id}")

# Load into DataFrames
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with {len(df)} records and columns: {df.columns.tolist()}")

# For further steps, let's pick the first (main) record set
if len(record_sets_ids) >= 1:
    main_record_set_id = record_sets_ids[0]
    print(f"\nColumn names in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping/categorizing data. All fields are referenced using their `@id` or column names identified above.

In [ ]:
import numpy as np

# Choose the main DataFrame and preview numeric columns
df = dataframes[main_record_set_id]

# Try to infer some numeric fields (columns with float or int types)
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields detected: {numeric_candidates}")

# For demonstration, pick the first available numeric field
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise Exception("No numeric field found for EDA. Please inspect the DataFrame above.")

# Set a threshold (mean or fixed for demonstration)
if len(df[numeric_field].dropna()) > 0:
    threshold = df[numeric_field].mean()
else:
    threshold = 0

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records in '{main_record_set_id}' with '{numeric_field}' > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field (first string/object field)
group_field_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
for candidate in group_field_candidates:
    if candidate != numeric_field:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by '{group_field}':")
    display(grouped_df.head())
else:
    print("No categorical group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Here, we plot the distribution of the main numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of '{numeric_field}' in main record set")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If there are at least two numeric fields, show correlation
if len(numeric_candidates) > 1:
    plt.figure(figsize=(6,6))
    sns.scatterplot(data=df, x=numeric_candidates[0], y=numeric_candidates[1])
    plt.title(f"Scatter: {numeric_candidates[0]} vs {numeric_candidates[1]}")
    plt.show()


## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset by loading metadata, listing available record sets and fields (uniquely using their `@id`), converting records into pandas DataFrames, and conducting basic EDA—filtering, normalization, grouping, and visualization.

Key findings should be derived from further domain-specific analysis on protein abundance, modifications, and other features contained in this mass spectrometry dataset.